In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gradio as gr

# 1. Load CSV
def load_csv(file):
    df = pd.read_csv(file.name)
    columns = df.columns.tolist()
    
    return df, df.head(), gr.update(choices=columns, value=columns[0])

# 2. Summary
def get_summary(df):
    return df.describe(include='all').to_string()

# 3. Univariate Plot
def univariate_plot(df, column):
    plt.figure()

    if pd.api.types.is_numeric_dtype(df[column]):
        sns.histplot(df[column], kde=True)
    else:
        sns.countplot(x=df[column])

    plt.title(f"Distribution of {column}")
    return plt.gcf()

# 4. Bivariate Plot
def bivariate_plot(df, column):
    if "Hired" not in df.columns:
        return None

    plt.figure()

    if pd.api.types.is_numeric_dtype(df[column]):
        sns.boxplot(x="Hired", y=column, data=df)
    else:
        sns.countplot(x=column, hue="Hired", data=df)

    plt.title(f"{column} vs Hiring")
    return plt.gcf()

# 5. Correlation
def correlation_plot(df):
    plt.figure()
    sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm")
    return plt.gcf()

# 6. Feature Engineering
def feature_engineering(df):
    df = df.copy()

    if "YearsExperience" in df.columns:
        df["ExperienceLevel"] = pd.cut(
            df["YearsExperience"],
            bins=[0,5,10,20,50],
            labels=["Junior","Mid","Senior","Expert"]
        )

    if "SkillsScore" in df.columns and "CertificationCount" in df.columns:
        df["SkillPerCert"] = df["SkillsScore"] / (df["CertificationCount"] + 1)

    return df.head().to_string()

# ================= UI ================= #

with gr.Blocks() as app:

    gr.Markdown("# 📊 Dynamic EDA App")

    state = gr.State()  # store dataframe

    file_input = gr.File(label="Upload CSV")
    preview = gr.Dataframe()
    column_dropdown = gr.Dropdown(label="Select Column")

    file_input.change(
        fn=load_csv,
        inputs=file_input,
        outputs=[state, preview, column_dropdown]
    )

    # Summary
    summary_output = gr.Textbox()
    gr.Button("Show Summary").click(
        fn=get_summary,
        inputs=state,
        outputs=summary_output
    )

    # Univariate
    uni_plot = gr.Plot()
    gr.Button("Univariate Plot").click(
        fn=univariate_plot,
        inputs=[state, column_dropdown],
        outputs=uni_plot
    )

    # Bivariate
    bi_plot = gr.Plot()
    gr.Button("Bivariate vs Hired").click(
        fn=bivariate_plot,
        inputs=[state, column_dropdown],
        outputs=bi_plot
    )

    # Correlation
    corr_plot = gr.Plot()
    gr.Button("Correlation Heatmap").click(
        fn=correlation_plot,
        inputs=state,
        outputs=corr_plot
    )

    # Feature Engineering
    feature_output = gr.Textbox()
    gr.Button("Feature Engineering").click(
        fn=feature_engineering,
        inputs=state,
        outputs=feature_output
    )

app.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
